# TP 6 – Combien va coûter cette course ? (30 min)

**Objectif** : prédire `fare_amount` à partir de la distance, de l'heure, du nombre de passagers et de l'arrondissement
de départ. Comparer une régression linéaire et un arbre de régression.

**Réussi si** : RMSE de l'arbre < 5 $ sur le test.

In [ ]:
# Se placer à la racine du dépôt : les chemins data/... sont relatifs à la racine
import os
while not os.path.isdir("data") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")
print("Répertoire de travail :", os.getcwd())

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler
spark = SparkSession.builder.appName("TP").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
zones = spark.read.csv("data/taxi_zone_lookup.csv", header=True, inferSchema=True)

from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator

### 1. Les données : ajouter `heure` et `Borough` (jointure avec les zones), garder les courses de distance > 0

Colonnes à garder : `trip_distance`, `heure`, `passenger_count`, `Borough`, `fare_amount`.

In [ ]:
df = (spark.read.parquet("data/trips.parquet")
      .filter(...)
      .withColumn("heure", ...)
      .join(...)
      .select(...))

### 2. Train / test : 80 / 20, `seed=42`

In [ ]:
train, test = ...
print(train.count(), test.count())

### 3. La préparation : `StringIndexer` (Borough → index, `handleInvalid="keep"`) → `OneHotEncoder` → `VectorAssembler`

In [ ]:
preparation = [
    ...
]

### 4. Un pipeline par modèle : `LinearRegression`, puis `DecisionTreeRegressor(maxDepth=6)`

Indice : `Pipeline(stages=preparation + [modele]).fit(train)`, puis `.transform(test)`.

In [ ]:
...

### 5. Comparer RMSE et R² sur le test

Indice : `RegressionEvaluator(labelCol="fare_amount", metricName="rmse")`. Lequel est le meilleur ? Pourquoi ? (indice : 70 $)

In [ ]:
...

### 6. Sauvegarder le meilleur pipeline, puis le recharger

Indice : `model.write().overwrite().save("out/modele_montant")`, `PipelineModel.load(...)`.

In [ ]:
from pyspark.ml import PipelineModel
...

### Bonus 1 – `GBTRegressor` fait-il mieux ?

In [ ]:
...

### Bonus 2 – « Tricher » : ajoutez `total_amount` aux features. Quel R² ? Pourquoi ce modèle est-il inutilisable ?

In [ ]:
...